# MedPredict s.r.l. — Previsione della Progressione del Diabete

**Caso d'uso aziendale:** Previsione della progressione del diabete in pazienti a rischio.

**Obiettivo:** Sviluppare un modello di regressione predittiva che, utilizzando dati clinici dei pazienti, fornisca previsioni accurate sulla progressione della malattia nel tempo.

**Dataset:** Diabetes dataset (scikit-learn) — 442 pazienti, 10 variabili cliniche, target quantitativo di progressione della malattia.

---

### Variabili indipendenti
| Feature | Descrizione |
|---------|-------------|
| Age | Età del paziente |
| Sex | Genere |
| BMI | Indice di massa corporea |
| BP | Pressione sanguigna media |
| S1 | Colesterolo sierico totale |
| S2 | Lipoproteine a bassa densità (LDL) |
| S3 | Lipoproteine ad alta densità (HDL) |
| S4 | Rapporto colesterolo totale / HDL |
| S5 | Trigliceridi (log) |
| S6 | Livello di glicemia |

**Target:** misura quantitativa della progressione del diabete a un anno dalla raccolta dati.

# Moduli

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline

# Metodi comuni

La funzione `print_correlation_matrix` visualizza la matrice di correlazione come una heatmap. Prende in input un DataFrame `df` e utilizza Seaborn per generare una heatmap della matrice di correlazione. I valori degli indici di Pearson vengono mostrati all'interno delle celle.

In [ ]:
def print_correlation_matrix(df):
    """
    Print correlation matrix as a heatmap.

    Args:
        df (DataFrame): Input DataFrame.

    Returns:
        None
    """
    plt.figure(figsize=(12, 10), dpi=100)
    sns.set_style("whitegrid")
    sns.heatmap(
        df.corr(),
        cmap="crest",
        cbar=True,
        square=True,
        yticklabels=df.columns,
        xticklabels=df.columns,
        annot=True,
        annot_kws={'size': 10},
        fmt='.2f'
    )
    plt.title("Matrice di Correlazione", fontsize=14)
    plt.tight_layout()
    plt.show()

La funzione `scale_features` esegue la standardizzazione delle features utilizzando `StandardScaler`. Applica il fit sui dati di training e trasforma sia training che test, evitando data leakage.

In [ ]:
def scale_features(X_train, X_test):
    """
    Scale features using StandardScaler.

    Args:
        X_train (array-like): Training data.
        X_test (array-like): Test data.

    Returns:
        Tuple: (X_train_scaled, X_test_scaled, scaler)
    """
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled, scaler

La funzione `evaluate_regression` valuta le prestazioni di un modello di regressione. Calcola e stampa MSE, RMSE, MAE e R² sia per i dati di training che di test, permettendo di identificare eventuali fenomeni di overfitting.

In [ ]:
def evaluate_regression(model_name, y_train, y_test, y_pred_train, y_pred_test):
    """
    Evaluate regression model performance.

    Args:
        model_name (str): Name of the model.
        y_train, y_test (array-like): True values.
        y_pred_train, y_pred_test (array-like): Predicted values.

    Returns:
        dict: Dictionary with test metrics.
    """
    metrics = {}
    for split, y_true, y_pred in [("TRAIN", y_train, y_pred_train), ("TEST", y_test, y_pred_test)]:
        mse  = mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        mae  = mean_absolute_error(y_true, y_pred)
        r2   = r2_score(y_true, y_pred)
        print(f"--- {model_name} | {split} ---")
        print(f"  MSE  : {mse:.2f}")
        print(f"  RMSE : {rmse:.2f}")
        print(f"  MAE  : {mae:.2f}")
        print(f"  R²   : {r2:.4f}")
        if split == "TEST":
            metrics = {"MSE": mse, "RMSE": rmse, "MAE": mae, "R2": r2}
    return metrics

La funzione `plot_predictions` visualizza i valori predetti rispetto ai valori reali tramite uno scatter plot, utile per valutare visivamente la qualità del modello. Una retta ideale al 45° indica previsioni perfette.

In [ ]:
def plot_predictions(y_test, y_pred, model_name):
    """
    Plot predicted vs actual values.

    Args:
        y_test (array-like): True values.
        y_pred (array-like): Predicted values.
        model_name (str): Name of the model for the title.

    Returns:
        None
    """
    plt.figure(figsize=(7, 6), dpi=100)
    plt.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidths=0.4)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Previsione perfetta')
    plt.xlabel('Valori Reali')
    plt.ylabel('Valori Predetti')
    plt.title(f'Predetto vs Reale — {model_name}')
    plt.legend()
    plt.tight_layout()
    plt.show()

La funzione `plot_residuals` visualizza i residui (errori) del modello in funzione dei valori predetti. Un buon modello presenta residui distribuiti casualmente attorno allo zero, senza pattern sistematici.

In [ ]:
def plot_residuals(y_test, y_pred, model_name):
    """
    Plot residuals of the regression model.

    Args:
        y_test (array-like): True values.
        y_pred (array-like): Predicted values.
        model_name (str): Name of the model.

    Returns:
        None
    """
    residuals = y_test - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

    axes[0].scatter(y_pred, residuals, alpha=0.6, edgecolors='k', linewidths=0.4)
    axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
    axes[0].set_xlabel('Valori Predetti')
    axes[0].set_ylabel('Residui')
    axes[0].set_title(f'Residui vs Predetti — {model_name}')

    axes[1].hist(residuals, bins=25, edgecolor='black', color='steelblue')
    axes[1].set_xlabel('Residuo')
    axes[1].set_ylabel('Frequenza')
    axes[1].set_title(f'Distribuzione Residui — {model_name}')

    plt.tight_layout()
    plt.show()

# Caricamento del Dataset

Il dataset Diabetes di scikit-learn viene caricato come DataFrame Pandas, aggiungendo la colonna `target` che rappresenta la progressione quantitativa della malattia misurata un anno dopo la raccolta delle feature cliniche. Le variabili sono già normalizzate a media zero e deviazione standard unitaria nella versione originale; lavoreremo però con i valori originali estratti dal campo `data` per avere piena visibilità sul pre-processing.

In [ ]:
diabetes = load_diabetes()

df = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
df['target'] = diabetes.target

print(f"Dimensioni dataset: {df.shape}")
print(f"Feature: {list(diabetes.feature_names)}")
df.head()

# Analisi Esplorativa dei Dati (EDA)

## Statistiche descrittive

Calcoliamo le statistiche descrittive di base per comprendere la distribuzione delle variabili cliniche. Valutiamo media, deviazione standard, valori minimi e massimi, nonché i percentili principali.

In [ ]:
df.describe()

## Controllo valori mancanti

Verifichiamo l'assenza di valori nulli nel dataset, requisito fondamentale per addestrare correttamente il modello di regressione.

In [ ]:
null_counts = df.isnull().sum()
print("Valori nulli per colonna:")
print(null_counts)
print(f"\nTotale valori nulli: {null_counts.sum()}")

## Distribuzione della variabile target

Analizziamo la distribuzione del target, ovvero la progressione quantitativa del diabete. Una distribuzione approssimativamente normale è favorevole per i modelli di regressione lineare.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

axes[0].hist(df['target'], bins=30, edgecolor='black', color='steelblue')
axes[0].set_xlabel('Progressione Diabete (target)')
axes[0].set_ylabel('Frequenza')
axes[0].set_title('Distribuzione del Target')

axes[1].boxplot(df['target'], vert=True)
axes[1].set_ylabel('Progressione Diabete (target)')
axes[1].set_title('Boxplot del Target')

plt.tight_layout()
plt.show()

print(f"Media target: {df['target'].mean():.2f}")
print(f"Mediana target: {df['target'].median():.2f}")
print(f"Deviazione standard target: {df['target'].std():.2f}")

## Distribuzione delle variabili cliniche

Visualizziamo la distribuzione di ciascuna variabile indipendente tramite istogrammi. Questo ci permette di identificare eventuali asimmetrie, code lunghe o distribuzioni bimodali che potrebbero influenzare le prestazioni del modello.

In [ ]:
feature_cols = diabetes.feature_names
fig, axes = plt.subplots(2, 5, figsize=(20, 8), dpi=100)
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].hist(df[col], bins=20, edgecolor='black', color='teal', alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel('Valore')
    axes[i].set_ylabel('Frequenza')

plt.suptitle('Distribuzione delle Variabili Cliniche', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Scatter plot: Feature vs Target

Visualizziamo la relazione lineare tra ciascuna feature e la variabile target. Le variabili con correlazione più forte (scatterplot con forma allungata) saranno candidate privilegiate per il modello.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8), dpi=100)
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].scatter(df[col], df['target'], alpha=0.5, edgecolors='k', linewidths=0.2, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Target')
    axes[i].set_title(f'{col} vs Target')

plt.suptitle('Relazione Feature-Target', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Matrice di correlazione

La matrice di correlazione di Pearson mostra le relazioni lineari tra tutte le variabili. I valori vicini a +1 o -1 indicano forte correlazione, mentre valori vicini a 0 indicano assenza di correlazione lineare. Particolare attenzione va posta all'ultima riga/colonna che mostra la correlazione di ciascuna feature con il target.

In [ ]:
print_correlation_matrix(df)

Estraiamo e ordiniamo le correlazioni con la variabile target per identificare rapidamente le feature più predittive.

In [ ]:
corr_with_target = df.corr()['target'].drop('target').sort_values(key=abs, ascending=False)

plt.figure(figsize=(9, 5), dpi=100)
colors = ['steelblue' if v > 0 else 'tomato' for v in corr_with_target.values]
plt.bar(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Feature Clinica')
plt.ylabel('Correlazione con Target')
plt.title('Correlazione delle Feature con la Progressione del Diabete')
plt.tight_layout()
plt.show()

print(corr_with_target)

# Pulizia e Pre-processing dei Dati

La variabile `sex` è codificata come -1 o 1 nella versione normalizzata del dataset. Nell'originale è già numerica. Non sono presenti variabili categoriche con valori stringa, pertanto non è richiesto encoding aggiuntivo.

Verifichiamo i tipi di dato e valutiamo eventuali outlier tramite boxplot.

In [ ]:
print("Tipi di dato:")
print(df.dtypes)
print(f"\nShape: {df.shape}")

## Analisi degli outlier

Utilizziamo boxplot per identificare valori anomali. I punti oltre i baffi del boxplot (1.5×IQR) sono potenziali outlier che potrebbero influenzare le previsioni del modello.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8), dpi=100)
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    axes[i].boxplot(df[col])
    axes[i].set_title(col)
    axes[i].set_ylabel('Valore')

plt.suptitle('Boxplot delle Variabili Cliniche — Analisi Outlier', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Selezione delle Variabili

## Selezione tramite Regressione Lasso

La Regressione Lasso (L1) penalizza la somma dei valori assoluti dei coefficienti, portando a coefficienti esattamente zero per le feature meno rilevanti. Questo la rende un ottimo strumento per la selezione automatica delle variabili.

Utilizziamo GridSearchCV per trovare il valore ottimale del parametro di regolarizzazione `alpha`.

In [ ]:
X_all = df.drop('target', axis=1).values
y_all = df['target'].values

X_train_sel, X_test_sel, y_train_sel, y_test_sel = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)
X_train_sc, X_test_sc, _ = scale_features(X_train_sel, X_test_sel)

alphas = np.logspace(-4, 1, 50)
lasso_cv = GridSearchCV(
    Lasso(max_iter=10000),
    param_grid={'alpha': alphas},
    cv=5,
    scoring='r2'
)
lasso_cv.fit(X_train_sc, y_train_sel)

best_alpha = lasso_cv.best_params_['alpha']
print(f"Alpha ottimale per Lasso: {best_alpha:.6f}")
print(f"R² cross-validation: {lasso_cv.best_score_:.4f}")

In [ ]:
lasso_best = Lasso(alpha=best_alpha, max_iter=10000)
lasso_best.fit(X_train_sc, y_train_sel)

coef_df = pd.DataFrame({
    'Feature': diabetes.feature_names,
    'Coefficiente': lasso_best.coef_
}).sort_values('Coefficiente', key=abs, ascending=False)

print("Coefficienti Lasso (ordinati per importanza):")
print(coef_df.to_string(index=False))

plt.figure(figsize=(10, 5), dpi=100)
colors = ['steelblue' if v > 0 else 'tomato' for v in coef_df['Coefficiente']]
plt.bar(coef_df['Feature'], coef_df['Coefficiente'], color=colors, edgecolor='black')
plt.axhline(0, color='black', linewidth=0.8)
plt.xlabel('Feature Clinica')
plt.ylabel('Coefficiente Lasso')
plt.title(f'Coefficienti Lasso (alpha={best_alpha:.4f})')
plt.tight_layout()
plt.show()

Selezioniamo le feature con coefficiente Lasso diverso da zero come variabili per i modelli successivi. Le feature con coefficiente azzerato da Lasso non contribuiscono significativamente alla predizione della progressione del diabete.

In [ ]:
selected_features = coef_df[coef_df['Coefficiente'] != 0]['Feature'].tolist()
eliminated_features = coef_df[coef_df['Coefficiente'] == 0]['Feature'].tolist()

print(f"Feature selezionate da Lasso ({len(selected_features)}): {selected_features}")
print(f"Feature eliminate da Lasso ({len(eliminated_features)}): {eliminated_features}")

df_selected = df[selected_features + ['target']]
print(f"\nShape dataset ridotto: {df_selected.shape}")

# Creazione del Modello di Regressione

## Preparazione dei dati

Dividiamo il dataset in set di training (80%) e test (20%) con `random_state=42` per garantire la riproducibilità. Standardizziamo le feature applicando `StandardScaler` solo sui dati di training per evitare data leakage verso il test set.

In [ ]:
X = df_selected.drop('target', axis=1).values
y = df_selected['target'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_scaled, X_test_scaled, final_scaler = scale_features(X_train, X_test)

print(f"Training set: {X_train.shape[0]} campioni")
print(f"Test set:     {X_test.shape[0]} campioni")
print(f"Feature usate: {X_train.shape[1]}")

## Addestramento e confronto dei modelli

Addestriamo e confrontiamo i seguenti modelli di regressione:

| Modello | Tipo di regolarizzazione |
|---------|-------------------------|
| Linear Regression | Nessuna (baseline) |
| Ridge | L2 |
| Lasso | L1 |
| ElasticNet | L1 + L2 |
| Gradient Boosting | Ensemble / boosting |

I risultati verranno confrontati utilizzando MSE, RMSE e R².

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=best_alpha, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=best_alpha, l1_ratio=0.5, max_iter=10000),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred_train = model.predict(X_train_scaled)
    y_pred_test  = model.predict(X_test_scaled)
    metrics = evaluate_regression(name, y_train, y_test, y_pred_train, y_pred_test)
    results[name] = metrics
    print()

## Confronto visivo dei modelli

Visualizziamo il confronto delle metriche di test (MSE e R²) per tutti i modelli addestrati. Il modello migliore è quello con R² più alto (più vicino a 1) e MSE più basso.

In [ ]:
results_df = pd.DataFrame(results).T.reset_index()
results_df.columns = ['Modello', 'MSE', 'RMSE', 'MAE', 'R2']
results_df = results_df.sort_values('R2', ascending=False)

print("Confronto metriche di test:")
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

axes[0].barh(results_df['Modello'], results_df['R2'], color='steelblue', edgecolor='black')
axes[0].set_xlabel('R²')
axes[0].set_title('R² — Confronto Modelli (Test Set)')
axes[0].axvline(x=0, color='black', linewidth=0.8)

axes[1].barh(results_df['Modello'], results_df['RMSE'], color='tomato', edgecolor='black')
axes[1].set_xlabel('RMSE')
axes[1].set_title('RMSE — Confronto Modelli (Test Set)')

plt.tight_layout()
plt.show()

# Valutazione del Modello Finale

Selezioniamo il modello con il maggiore R² sul test set come modello finale da esportare. Analizziamo in dettaglio le sue previsioni tramite scatter plot e distribuzione dei residui.

In [ ]:
best_model_name = results_df.iloc[0]['Modello']
best_model = models[best_model_name]

print(f"Modello selezionato: {best_model_name}")
print(f"R² Test: {results_df.iloc[0]['R2']:.4f}")
print(f"RMSE Test: {results_df.iloc[0]['RMSE']:.2f}")

y_pred_final = best_model.predict(X_test_scaled)

plot_predictions(y_test, y_pred_final, best_model_name)
plot_residuals(y_test, y_pred_final, best_model_name)

## Validazione incrociata (Cross-Validation)

Eseguiamo una cross-validation a 5 fold sull'intero dataset per ottenere una stima più robusta e meno dipendente dalla singola suddivisione train/test. I valori di R² per ogni fold ci mostrano la stabilità del modello.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', best_model)
])

cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')

print(f"R² per ogni fold: {np.round(cv_scores, 4)}")
print(f"R² medio: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

plt.figure(figsize=(8, 4), dpi=100)
plt.bar(range(1, 6), cv_scores, color='steelblue', edgecolor='black')
plt.axhline(cv_scores.mean(), color='red', linestyle='--', label=f'Media R²={cv_scores.mean():.3f}')
plt.xlabel('Fold')
plt.ylabel('R²')
plt.title(f'Cross-Validation R² — {best_model_name}')
plt.legend()
plt.tight_layout()
plt.show()

## Confronto con baseline

Confrontiamo il modello finale con due baseline naive:
- **Mean baseline**: predice sempre la media del training set
- **Median baseline**: predice sempre la mediana del training set

Un modello valido deve significativamente superare queste baseline.

In [ ]:
mean_pred  = np.full_like(y_test, y_train.mean(), dtype=float)
median_pred = np.full_like(y_test, np.median(y_train), dtype=float)

print("=== Baseline: Media ===")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, mean_pred)):.2f}")
print(f"  R²:   {r2_score(y_test, mean_pred):.4f}")

print("\n=== Baseline: Mediana ===")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, median_pred)):.2f}")
print(f"  R²:   {r2_score(y_test, median_pred):.4f}")

print(f"\n=== {best_model_name} ===")
print(f"  RMSE: {results_df.iloc[0]['RMSE']:.2f}")
print(f"  R²:   {results_df.iloc[0]['R2']:.4f}")

# Esportazione del Modello

Il modello finale viene esportato tramite `pickle` insieme allo scaler e alla lista delle feature selezionate. Questo permette di ricaricare il modello in produzione e applicarlo su nuovi dati clinici, garantendo che lo stesso pre-processing venga applicato in fase di inferenza.

In [ ]:
model_artifact = {
    'model': best_model,
    'scaler': final_scaler,
    'selected_features': selected_features,
    'model_name': best_model_name,
    'test_r2': float(results_df.iloc[0]['R2']),
    'test_rmse': float(results_df.iloc[0]['RMSE']),
}

output_path = 'medpredict_diabetes_model.pkl'

with open(output_path, 'wb') as f:
    pickle.dump(model_artifact, f)

print(f"Modello salvato in: {output_path}")
print(f"Modello: {best_model_name}")
print(f"Feature utilizzate: {selected_features}")
print(f"R² Test: {model_artifact['test_r2']:.4f}")
print(f"RMSE Test: {model_artifact['test_rmse']:.2f}")

Verifichiamo che il modello esportato sia correttamente ricaricabile e produca le stesse previsioni, simulando il comportamento in ambiente di produzione.

In [ ]:
with open(output_path, 'rb') as f:
    loaded_artifact = pickle.load(f)

loaded_model   = loaded_artifact['model']
loaded_scaler  = loaded_artifact['scaler']

X_test_loaded_scaled = loaded_scaler.transform(X_test)
y_pred_loaded = loaded_model.predict(X_test_loaded_scaled)

pred_match = np.allclose(y_pred_final, y_pred_loaded)
print(f"Previsioni identiche dopo il ricaricamento: {pred_match}")
print("Modello pronto per la piattaforma sanitaria MedPredict.")

# Conclusioni Finali

## Riepilogo del processo

Il progetto ha seguito l'intero ciclo di vita del machine learning, dalla raccolta dei dati all'esportazione del modello in produzione:

1. **Caricamento e analisi esplorativa**: Il dataset Diabetes di scikit-learn è stato esaminato nelle sue distribuzioni statistiche, identificando la natura approssimativamente normale del target e le relazioni tra le variabili cliniche.

2. **Selezione delle feature tramite Lasso**: La regressione Lasso ha automaticamente identificato le variabili più predittive, eliminando quelle con scarso potere esplicativo sulla progressione del diabete.

3. **Confronto multi-modello**: Sono stati addestrati e confrontati cinque modelli di regressione (Linear Regression, Ridge, Lasso, ElasticNet, Gradient Boosting), selezionando il migliore in base al R² sul test set.

4. **Validazione robusta**: La cross-validation a 5 fold ha confermato la stabilità del modello selezionato, mostrando R² consistente su tutti i fold.

5. **Confronto con baseline**: Il modello supera significativamente le baseline naive (media e mediana), confermando il valore aggiunto del machine learning per questo problema clinico.

6. **Esportazione**: Il modello è stato serializzato con `pickle` insieme allo scaler e ai metadati necessari per la produzione.

## Valore per MedPredict s.r.l.

Il sistema predittivo sviluppato consente ai medici di:
- Identificare in anticipo i pazienti con progressione rapida del diabete
- Personalizzare i piani terapeutici sulla base delle previsioni quantitative
- Ottimizzare il monitoraggio clinico, concentrando le risorse sui pazienti più a rischio
- Ridurre le complicazioni a lungo termine attraverso interventi preventivi mirati

## Limiti e sviluppi futuri

- Il dataset è limitato a 442 pazienti: l'integrazione di dataset clinici reali più ampi migliorerà la generalizzazione del modello.
- L'aggiunta di variabili temporali (storico clinico del paziente) potrebbe incrementare significativamente la capacità predittiva.
- L'implementazione di un sistema di monitoraggio del modello in produzione (data drift, performance degradation) è raccomandata prima del deploy clinico.